# 05B-QC — Finalize Temporal Features

**FYP:** Developing a Semi-Dynamic Landslide Susceptibility Decision-Support Dashboard Using Machine Learning for Malaysia

This notebook finalizes the outputs from `05B_temporal_feature_extraction.ipynb`.

It does **not** retrain any model yet.

---

# Current Stage-05B result

The temporal extraction produced:

- **189 dated, static-valid landslide cases**
- **378 date-matched controls**
- **567 total temporal samples**
- rainfall coverage: complete
- ERA5-Land soil moisture: mostly complete
- historical land cover: mixed source / incomplete
- waterway distance: complete
- historical roads: not yet run

---

# Purpose

This notebook:

1. harmonises GLC_FCS30D and Dynamic World land-cover classes;
2. preserves the original raw codes and source;
3. treats 2015 Dynamic World as partial-year / low-confidence;
4. flags water-background controls as invalid for terrestrial susceptibility modelling;
5. creates model-specific readiness flags;
6. preserves matched case-control groups;
7. creates separate datasets for:
   - rainfall model
   - rainfall + soil-moisture model
   - land-cover model
   - full temporal-enhanced model

---

# Important rule

Missing temporal features are **not imputed with invented values**.

Instead, the notebook creates separate model-ready subsets according to which features are actually available.


In [1]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

print("pandas:", pd.__version__)


pandas: 3.0.2


## 1. Project paths

In [2]:
PROJECT_ROOT = Path(r"C:\Landslide_FYP")

DEV_DIR = PROJECT_ROOT / "Landslide_FYP_Development"

INPUT_DIR = (
    DEV_DIR
    / "03_derived_features"
    / "temporal_features"
)

INPUT_PATH = (
    INPUT_DIR
    / "temporal_feature_samples.csv"
)

OUTPUT_DIR = (
    DEV_DIR
    / "03_derived_features"
    / "temporal_features_final"
)

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

if not INPUT_PATH.exists():
    raise FileNotFoundError(
        f"Temporal feature table not found:\\n{INPUT_PATH}"
    )

print("Input:", INPUT_PATH)
print("Output:", OUTPUT_DIR)


Input: C:\Landslide_FYP\Landslide_FYP_Development\03_derived_features\temporal_features\temporal_feature_samples.csv
Output: C:\Landslide_FYP\Landslide_FYP_Development\03_derived_features\temporal_features_final


## 2. Load Stage-05B temporal feature table

In [3]:
df = pd.read_csv(
    INPUT_PATH
)

df["reference_date"] = pd.to_datetime(
    df["reference_date"],
    errors="coerce"
)

df["reference_year"] = pd.to_numeric(
    df["reference_year"],
    errors="coerce"
)

print("Rows:", len(df))

display(
    df["temporal_role"]
    .value_counts()
    .rename_axis("temporal_role")
    .reset_index(name="records")
)

print(
    "Matched groups:",
    df["matched_positive_id"].nunique()
)


Rows: 567


,temporal_role,records
0,matched_control,378
1,case,189


Matched groups: 189


# PART A — LAND-COVER HARMONISATION

## 3. Harmonised classification

Both products are mapped into the following common classes:

| Code | Harmonised class |
|---:|---|
| 1 | FOREST |
| 2 | SHRUB |
| 3 | GRASS |
| 4 | CROPLAND |
| 5 | BUILT |
| 6 | WETLAND |
| 7 | BARE |
| 8 | WATER |
| 9 | OTHER |

The raw source code is preserved separately.

### GLC_FCS30D groups

- cropland: 10, 11, 12, 20
- forest: 51–92
- shrub: 120–122
- grass: 130
- wetland: 181–187
- built: 190
- bare/sparse: 150, 152, 153, 200–202
- water: 210
- other: 140, 220
- fill: 0, 250 → missing

### Dynamic World

- 0 water
- 1 trees
- 2 grass
- 3 flooded vegetation
- 4 crops
- 5 shrub/scrub
- 6 built
- 7 bare
- 8 snow/ice


In [4]:
HARMONISED_NAME_TO_CODE = {
    "FOREST": 1,
    "SHRUB": 2,
    "GRASS": 3,
    "CROPLAND": 4,
    "BUILT": 5,
    "WETLAND": 6,
    "BARE": 7,
    "WATER": 8,
    "OTHER": 9,
}


GLC_GROUPS = {
    "CROPLAND": {
        10, 11, 12, 20
    },

    "FOREST": {
        51, 52,
        61, 62,
        71, 72,
        81, 82,
        91, 92
    },

    "SHRUB": {
        120, 121, 122
    },

    "GRASS": {
        130
    },

    "WETLAND": {
        181, 182, 183,
        184, 185, 186, 187
    },

    "BUILT": {
        190
    },

    "BARE": {
        150, 152, 153,
        200, 201, 202
    },

    "WATER": {
        210
    },

    "OTHER": {
        140, 220
    }
}


DW_MAP = {
    0: "WATER",
    1: "FOREST",
    2: "GRASS",
    3: "WETLAND",
    4: "CROPLAND",
    5: "SHRUB",
    6: "BUILT",
    7: "BARE",
    8: "OTHER",
}


def harmonise_landcover(
    source,
    raw_code
):

    if pd.isna(source) or pd.isna(raw_code):
        return pd.NA, np.nan

    try:
        code = int(
            round(
                float(raw_code)
            )
        )
    except Exception:
        return pd.NA, np.nan


    if source == "GLC_FCS30D":

        if code in {0, 250}:
            return pd.NA, np.nan

        for category, codes in GLC_GROUPS.items():

            if code in codes:

                return (
                    category,
                    HARMONISED_NAME_TO_CODE[
                        category
                    ]
                )

        return pd.NA, np.nan


    if source == "Dynamic_World":

        category = DW_MAP.get(
            code
        )

        if category is None:
            return pd.NA, np.nan

        return (
            category,
            HARMONISED_NAME_TO_CODE[
                category
            ]
        )


    return pd.NA, np.nan


harmonised = df.apply(
    lambda row:
        harmonise_landcover(
            row.get(
                "landcover_source"
            ),
            row.get(
                "landcover_code_raw"
            )
        ),
    axis=1
)


df[
    "landcover_harmonised"
] = [
    item[0]
    for item in harmonised
]


df[
    "landcover_harmonised_code"
] = [
    item[1]
    for item in harmonised
]


display(
    df[
        [
            "landcover_source",
            "landcover_code_raw",
            "landcover_harmonised",
            "landcover_harmonised_code"
        ]
    ]
    .drop_duplicates()
    .sort_values(
        [
            "landcover_source",
            "landcover_code_raw"
        ],
        na_position="last"
    )
)


,landcover_source,landcover_code_raw,landcover_harmonised,landcover_harmonised_code
381,Dynamic_World,0.0,WATER,8.0
30,Dynamic_World,1.0,FOREST,1.0
84,Dynamic_World,2.0,GRASS,3.0
223,Dynamic_World,4.0,CROPLAND,4.0
6,Dynamic_World,6.0,BUILT,5.0
478,Dynamic_World,8.0,OTHER,9.0
0,GLC_FCS30D,10.0,CROPLAND,4.0
1,GLC_FCS30D,11.0,CROPLAND,4.0
94,GLC_FCS30D,12.0,CROPLAND,4.0
49,GLC_FCS30D,20.0,CROPLAND,4.0


## 4. Land-cover source reliability

Rules:

- `GLC_FCS30D` valid pixel → `HIGH`
- Dynamic World 2016+ valid annual result → `HIGH`
- Dynamic World 2015 → `LOW_PARTIAL_YEAR`
- no land-cover pixel → `MISSING`

2015 is flagged because Dynamic World begins part-way through 2015, so it does not represent a complete annual observation period.


In [5]:
def landcover_reliability(
    row
):

    source = row.get(
        "landcover_source"
    )

    year = row.get(
        "reference_year"
    )

    harmonised = row.get(
        "landcover_harmonised"
    )


    if pd.isna(
        harmonised
    ):
        return "MISSING"


    if source == "GLC_FCS30D":
        return "HIGH"


    if source == "Dynamic_World":

        if pd.notna(year) and int(year) == 2015:
            return "LOW_PARTIAL_YEAR"

        return "HIGH"


    return "UNKNOWN"


df[
    "landcover_reliability"
] = df.apply(
    landcover_reliability,
    axis=1
)


display(
    df.groupby(
        [
            "reference_year",
            "landcover_source",
            "landcover_reliability"
        ],
        dropna=False
    )
    .size()
    .reset_index(
        name="records"
    )
    .sort_values(
        [
            "reference_year",
            "landcover_source"
        ]
    )
)


,reference_year,landcover_source,landcover_reliability,records
0,2007.0,GLC_FCS30D,HIGH,6
1,2008.0,GLC_FCS30D,HIGH,12
2,2009.0,GLC_FCS30D,HIGH,21
3,2010.0,GLC_FCS30D,HIGH,15
4,2011.0,GLC_FCS30D,HIGH,42
5,2012.0,GLC_FCS30D,HIGH,27
6,2013.0,GLC_FCS30D,HIGH,48
7,2014.0,GLC_FCS30D,HIGH,69
8,2015.0,Dynamic_World,LOW_PARTIAL_YEAR,7
9,2015.0,NaN,MISSING,38


## 5. Identify invalid terrestrial land-cover controls

A background control classified as `WATER` is not a suitable terrestrial pseudo-absence/control for a landslide susceptibility model.

Cases are not automatically deleted if classified as water; such a result is instead flagged for review because it may indicate positional or land-cover mismatch.


In [6]:
df[
    "landcover_is_water"
] = (
    df[
        "landcover_harmonised"
    ]
    .eq(
        "WATER"
    )
)


df[
    "water_background_invalid"
] = (
    df[
        "temporal_role"
    ]
    .eq(
        "matched_control"
    )
    &
    df[
        "landcover_is_water"
    ]
)


df[
    "water_case_review"
] = (
    df[
        "temporal_role"
    ]
    .eq(
        "case"
    )
    &
    df[
        "landcover_is_water"
    ]
)


print(
    "Water background controls:",
    int(
        df[
            "water_background_invalid"
        ].sum()
    )
)

print(
    "Water positive cases:",
    int(
        df[
            "water_case_review"
        ].sum()
    )
)


display(
    df.loc[
        df[
            "water_background_invalid"
        ]
        |
        df[
            "water_case_review"
        ],
        [
            "sample_id",
            "temporal_role",
            "matched_positive_id",
            "reference_year",
            "landcover_source",
            "landcover_code_raw",
            "landcover_harmonised",
            "longitude",
            "latitude"
        ]
    ]
)


Water background controls: 3
Water positive cases: 0


,sample_id,temporal_role,matched_positive_id,reference_year,landcover_source,landcover_code_raw,landcover_harmonised,longitude,latitude
381,BG_00173,matched_control,POS_0156,2017.0,Dynamic_World,0.0,WATER,100.788496,5.934705
452,BG_00959,matched_control,POS_0218,2016.0,Dynamic_World,0.0,WATER,102.842796,5.138177
510,BG_02754,matched_control,POS_0254,2021.0,Dynamic_World,0.0,WATER,117.214739,5.827910


# PART B — FEATURE VALIDITY FLAGS

## 6. Rainfall readiness


In [7]:
rain_cols = [
    "rain_1d_mm",
    "rain_3d_mm",
    "rain_7d_mm",
    "rain_30d_mm",
]


df[
    "rainfall_complete"
] = (
    df[
        rain_cols
    ]
    .notna()
    .all(axis=1)
)


df[
    "rainfall_logical"
] = (
    df["rain_1d_mm"].le(
        df["rain_3d_mm"]
    )
    &
    df["rain_3d_mm"].le(
        df["rain_7d_mm"]
    )
    &
    df["rain_7d_mm"].le(
        df["rain_30d_mm"]
    )
)


df[
    "rainfall_valid"
] = (
    df[
        "rainfall_complete"
    ]
    &
    df[
        "rainfall_logical"
    ]
)


print(
    "Rainfall valid:",
    int(
        df[
            "rainfall_valid"
        ].sum()
    ),
    "/",
    len(df)
)


Rainfall valid: 567 / 567


## 7. ERA5-Land soil-moisture readiness


In [8]:
soil_moisture_cols = [
    "soil_moisture_0_7cm_7d_mean",
    "soil_moisture_7_28cm_7d_mean",
]


df[
    "soil_moisture_complete"
] = (
    df[
        soil_moisture_cols
    ]
    .notna()
    .all(axis=1)
)


print(
    "ERA5 soil moisture complete:",
    int(
        df[
            "soil_moisture_complete"
        ].sum()
    ),
    "/",
    len(df)
)


ERA5 soil moisture complete: 534 / 567


## 8. Land-cover readiness

Strict land-cover readiness requires:

- harmonised class available
- reliability = HIGH
- not an invalid water background control

Thus 2015 Dynamic World is excluded from the strict land-cover model unless a better historical land-cover source is later supplied.


In [9]:
df[
    "landcover_valid_strict"
] = (
    df[
        "landcover_harmonised"
    ]
    .notna()
    &
    df[
        "landcover_reliability"
    ]
    .eq(
        "HIGH"
    )
    &
    ~df[
        "water_background_invalid"
    ]
)


df[
    "landcover_valid_relaxed"
] = (
    df[
        "landcover_harmonised"
    ]
    .notna()
    &
    ~df[
        "water_background_invalid"
    ]
)


print(
    "Strict land-cover valid:",
    int(
        df[
            "landcover_valid_strict"
        ].sum()
    ),
    "/",
    len(df)
)


print(
    "Relaxed land-cover valid:",
    int(
        df[
            "landcover_valid_relaxed"
        ].sum()
    ),
    "/",
    len(df)
)


Strict land-cover valid: 493 / 567
Relaxed land-cover valid: 500 / 567


## 9. Waterway and historical-road readiness


In [10]:
df[
    "waterway_available"
] = (
    pd.to_numeric(
        df[
            "distance_waterway_current_m"
        ],
        errors="coerce"
    )
    .notna()
)


df[
    "historical_road_available"
] = (
    pd.to_numeric(
        df[
            "distance_road_historical_m"
        ],
        errors="coerce"
    )
    .notna()
)


print(
    "Waterway available:",
    int(
        df[
            "waterway_available"
        ].sum()
    )
)


print(
    "Historical road available:",
    int(
        df[
            "historical_road_available"
        ].sum()
    )
)


Waterway available: 567
Historical road available: 0


# PART C — MATCHED-GROUP VALIDATION

## 10. Helper for complete 1:2 matched groups

A valid group must contain:

- exactly 1 case
- exactly 2 matched controls
- all 3 rows satisfy the requested feature-validity rule

This prevents feature-specific missingness from silently breaking the matching design.


In [11]:
def identify_complete_matched_groups(
    dataframe,
    row_valid_column,
    expected_controls=2
):

    valid_group_ids = []

    audit_rows = []

    for group_id, group in dataframe.groupby(
        "matched_positive_id"
    ):

        n_case = int(
            group[
                "temporal_role"
            ]
            .eq(
                "case"
            )
            .sum()
        )

        n_control = int(
            group[
                "temporal_role"
            ]
            .eq(
                "matched_control"
            )
            .sum()
        )

        all_rows_valid = bool(
            group[
                row_valid_column
            ]
            .fillna(False)
            .all()
        )

        structure_valid = (
            n_case == 1
            and n_control == expected_controls
        )

        group_valid = (
            structure_valid
            and all_rows_valid
        )

        if group_valid:
            valid_group_ids.append(
                group_id
            )

        audit_rows.append({
            "matched_positive_id":
                group_id,

            "n_case":
                n_case,

            "n_control":
                n_control,

            "structure_valid":
                structure_valid,

            "all_rows_valid":
                all_rows_valid,

            "group_valid":
                group_valid,
        })

    return (
        valid_group_ids,
        pd.DataFrame(
            audit_rows
        )
    )


# PART D — MODEL-SPECIFIC DATASETS

## 11. Rainfall model

Because rainfall coverage is complete, this should retain essentially all matched groups.


In [12]:
rain_group_ids, rain_group_audit = (
    identify_complete_matched_groups(
        df,
        "rainfall_valid"
    )
)


rainfall_model = df.loc[
    df[
        "matched_positive_id"
    ]
    .isin(
        rain_group_ids
    )
].copy()


print(
    "Rainfall model groups:",
    len(
        rain_group_ids
    )
)

print(
    "Rainfall model rows:",
    len(
        rainfall_model
    )
)


rain_group_audit.to_csv(
    OUTPUT_DIR
    / "rainfall_matched_group_audit.csv",
    index=False
)


rainfall_model.to_csv(
    OUTPUT_DIR
    / "temporal_model_rainfall.csv",
    index=False
)


Rainfall model groups: 189
Rainfall model rows: 567


## 12. Rainfall + ERA5 soil-moisture model


In [13]:
df[
    "hydro_row_valid"
] = (
    df[
        "rainfall_valid"
    ]
    &
    df[
        "soil_moisture_complete"
    ]
)


hydro_group_ids, hydro_group_audit = (
    identify_complete_matched_groups(
        df,
        "hydro_row_valid"
    )
)


hydro_model = df.loc[
    df[
        "matched_positive_id"
    ]
    .isin(
        hydro_group_ids
    )
].copy()


print(
    "Hydro model groups:",
    len(
        hydro_group_ids
    )
)

print(
    "Hydro model rows:",
    len(
        hydro_model
    )
)


hydro_group_audit.to_csv(
    OUTPUT_DIR
    / "hydro_matched_group_audit.csv",
    index=False
)


hydro_model.to_csv(
    OUTPUT_DIR
    / "temporal_model_rainfall_soilmoisture.csv",
    index=False
)


Hydro model groups: 158
Hydro model rows: 474


## 13. Strict historical land-cover model


In [14]:
landcover_group_ids, landcover_group_audit = (
    identify_complete_matched_groups(
        df,
        "landcover_valid_strict"
    )
)


landcover_model = df.loc[
    df[
        "matched_positive_id"
    ]
    .isin(
        landcover_group_ids
    )
].copy()


print(
    "Strict land-cover groups:",
    len(
        landcover_group_ids
    )
)

print(
    "Strict land-cover rows:",
    len(
        landcover_model
    )
)


landcover_group_audit.to_csv(
    OUTPUT_DIR
    / "landcover_matched_group_audit.csv",
    index=False
)


landcover_model.to_csv(
    OUTPUT_DIR
    / "temporal_model_landcover.csv",
    index=False
)


Strict land-cover groups: 148
Strict land-cover rows: 444


## 14. Full enhanced temporal model

Required per row:

- rainfall valid
- ERA5 soil moisture complete
- strict harmonised land cover
- waterway distance available

Historical road distance is deliberately **not required** yet.


In [15]:
df[
    "full_temporal_row_valid"
] = (
    df[
        "rainfall_valid"
    ]
    &
    df[
        "soil_moisture_complete"
    ]
    &
    df[
        "landcover_valid_strict"
    ]
    &
    df[
        "waterway_available"
    ]
)


full_group_ids, full_group_audit = (
    identify_complete_matched_groups(
        df,
        "full_temporal_row_valid"
    )
)


full_temporal_model = df.loc[
    df[
        "matched_positive_id"
    ]
    .isin(
        full_group_ids
    )
].copy()


print(
    "Full temporal groups:",
    len(
        full_group_ids
    )
)

print(
    "Full temporal rows:",
    len(
        full_temporal_model
    )
)


full_group_audit.to_csv(
    OUTPUT_DIR
    / "full_temporal_matched_group_audit.csv",
    index=False
)


full_temporal_model.to_csv(
    OUTPUT_DIR
    / "temporal_model_full_enhanced.csv",
    index=False
)


Full temporal groups: 121
Full temporal rows: 363


# PART E — QC SUMMARIES

## 15. Land-cover harmonised distribution


In [16]:
landcover_distribution = (
    df.groupby(
        [
            "landcover_source",
            "landcover_harmonised"
        ],
        dropna=False
    )
    .size()
    .reset_index(
        name="records"
    )
    .sort_values(
        [
            "landcover_source",
            "records"
        ],
        ascending=[
            True,
            False
        ]
    )
)


display(
    landcover_distribution
)


landcover_distribution.to_csv(
    OUTPUT_DIR
    / "landcover_harmonised_distribution.csv",
    index=False
)


,landcover_source,landcover_harmonised,records
2,Dynamic_World,FOREST,192
0,Dynamic_World,BUILT,61
1,Dynamic_World,CROPLAND,3
3,Dynamic_World,GRASS,3
5,Dynamic_World,WATER,3
4,Dynamic_World,OTHER,1
8,GLC_FCS30D,FOREST,148
6,GLC_FCS30D,BUILT,40
7,GLC_FCS30D,CROPLAND,28
9,GLC_FCS30D,SHRUB,20


## 16. Model-subset summary


In [17]:
def model_counts(
    name,
    model_df
):

    return {
        "model_subset":
            name,

        "rows":
            len(
                model_df
            ),

        "matched_groups":
            model_df[
                "matched_positive_id"
            ]
            .nunique(),

        "cases":
            int(
                model_df[
                    "temporal_role"
                ]
                .eq(
                    "case"
                )
                .sum()
            ),

        "controls":
            int(
                model_df[
                    "temporal_role"
                ]
                .eq(
                    "matched_control"
                )
                .sum()
            ),
    }


model_subset_summary = pd.DataFrame(
    [
        model_counts(
            "Rainfall",
            rainfall_model
        ),

        model_counts(
            "Rainfall + ERA5 soil moisture",
            hydro_model
        ),

        model_counts(
            "Historical land cover",
            landcover_model
        ),

        model_counts(
            "Full temporal enhanced",
            full_temporal_model
        ),
    ]
)


display(
    model_subset_summary
)


model_subset_summary.to_csv(
    OUTPUT_DIR
    / "temporal_model_subset_summary.csv",
    index=False
)


,model_subset,rows,matched_groups,cases,controls
0,Rainfall,567,189,189,378
1,Rainfall + ERA5 soil moisture,474,158,158,316
2,Historical land cover,444,148,148,296
3,Full temporal enhanced,363,121,121,242


## 17. Year distribution of each final temporal subset


In [18]:
year_rows = []


for name, model_df in {
    "Rainfall":
        rainfall_model,

    "Rainfall + ERA5 soil moisture":
        hydro_model,

    "Historical land cover":
        landcover_model,

    "Full temporal enhanced":
        full_temporal_model,
}.items():

    case_df = model_df.loc[
        model_df[
            "temporal_role"
        ]
        .eq(
            "case"
        )
    ]

    counts = (
        case_df[
            "reference_year"
        ]
        .value_counts()
        .sort_index()
    )

    for year, count in counts.items():

        year_rows.append({
            "model_subset":
                name,

            "reference_year":
                int(
                    year
                ),

            "case_count":
                int(
                    count
                ),
        })


model_year_distribution = pd.DataFrame(
    year_rows
)


display(
    model_year_distribution
)


model_year_distribution.to_csv(
    OUTPUT_DIR
    / "temporal_model_year_distribution.csv",
    index=False
)


,model_subset,reference_year,case_count
0,Rainfall,2007,2
1,Rainfall,2008,4
2,Rainfall,2009,7
3,Rainfall,2010,5
4,Rainfall,2011,14
5,Rainfall,2012,9
6,Rainfall,2013,16
7,Rainfall,2014,23
8,Rainfall,2015,15
9,Rainfall,2016,17


## 18. Save fully annotated master temporal table


In [19]:
master_path = (
    OUTPUT_DIR
    / "temporal_feature_samples_final_annotated.csv"
)


df.to_csv(
    master_path,
    index=False
)


print(
    "Saved:",
    master_path
)


Saved: C:\Landslide_FYP\Landslide_FYP_Development\03_derived_features\temporal_features_final\temporal_feature_samples_final_annotated.csv


# Stage 05B-QC decision guide

After this notebook, do **not** force one universal temporal model.

Use the model-specific subsets:

### Rainfall model
Most complete temporal dataset.

### Rainfall + ERA5 soil-moisture model
Tests hydrological-state improvement.

### Historical land-cover model
Uses only matched groups with reliable harmonised land cover.

### Full temporal-enhanced model
Uses rainfall + soil moisture + historical land cover + current waterway distance.

Historical roads remain optional until ohsome coverage is assessed.

---

# Next development stage

If these subset sizes remain large enough, proceed to:

## `06_final_pseudoabsence_and_model_dataset.ipynb`

That stage will:

1. finalize the static-model pseudo-absence/control set;
2. preserve temporal matched-control subsets;
3. create spatial validation groups;
4. encode categorical predictors;
5. export the final Logistic Regression and Random Forest datasets.

Model training comes **after** that.


# Outputs to send for review

Please send:

```text
temporal_model_subset_summary.csv
temporal_model_year_distribution.csv
landcover_harmonised_distribution.csv
temporal_feature_samples_final_annotated.csv
```

The most important one is:

`temporal_model_subset_summary.csv`

because it tells us how many complete matched groups remain for each temporal experiment.
